# Duplicate group sorting and streaming

This tutorial is for readers familiar with catalog scans and bounded duplicate views. It shows how to sort duplicate groups, combine independent group/member bounds, process every observation incrementally, and close partially consumed streams.

Outline: (1) build disposable data, (2) compare bounded fetch and streams, (3) remove each bound explicitly, (4) sort groups, and (5) close resources safely.

## 1. Disposable setup

The fixture has nested paths, an empty-file group, and three duplicate groups totaling nine files and 16 potential redundant bytes.

In [ ]:
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog
from archiver.query import duplicate, size

workspace = TemporaryDirectory(prefix="archiver-004-4-1-")
base = Path(workspace.name)
root = base / "source"
root.mkdir()
for folder, payload, count in (("large/nested", b"AAAA", 4), ("small", b"BB", 3), ("empty", b"", 2), ("singeltons", b"1", 1)):
    directory = root / folder
    directory.mkdir(parents=True)
    for number in range(count):
        (directory / f"{number}.bin").write_bytes(payload)

catalog = Catalog.create(base / "catalog.sqlite")
catalog.scan_directory(root)

### Showing the simulated catalog

In [ ]:
# printing the catalog
from collections import Counter
from contextlib import closing

with closing(catalog.files(root).stream_all()) as stream:
    all_files = tuple(stream)

copy_count = Counter(file.content_id for file in all_files)

files_by_duplicate_count = sorted(
    all_files,
    key=lambda file: (
        -copy_count[file.content_id],  # largest groups first
        file.content_id.algorithm,     # keep equal-sized groups together
        file.content_id.digest,
        file.relative_path.as_posix(), # deterministic member order
    ),
)

print("All files, ordered by duplicate-group size\n",'-'*70)
for file in files_by_duplicate_count:
    count = copy_count[file.content_id]
    classification = "singleton" if count == 1 else f"{count} copies"
    print(
        f"{file.content_id.digest[:10]}..."
        f"  [{classification:>9}] "
        f"{file.relative_path.as_posix():<28} "
        f"{file.size_bytes:>3} bytes"
    )

## 2. Bounded fetch remains a reusable preview

`fetch()` retains the existing tuple result. Group metrics stay complete even though each member tuple is bounded.

In [ ]:
# created a bounded query
query = catalog.duplicates(root).sort(size.desc()).limit(2).member_limit(2)
# only the 1st 2 dup groups will be returned by fetch.
views = query.fetch()

for view in views:
    print(f"{view.content_id.digest[:10]}... " 
         f"{view.file_instance_count} Files, "
         f"{len(view.members)} Fetched")

In [ ]:

# since the sort is by size we have 4 and 2 bytes in the output just as indicated above
assert [view.size_bytes for view in views] == [4, 2]

# file counts in each dup group 
assert [view.file_instance_count for view in views] == [4, 3]

# member limit is 2 for all groups
assert all(len(view.members) <= 2 for view in views)

# summary will summ up all the info regardless of limits
summary = query.summary()
assert (summary.duplicate_content_group_count, # total dup groups = 3
        summary.duplicate_file_instance_count, # total files in dup groups = 4+3+2=9
        summary.potential_redundant_bytes) == (3, 9, 16) # redundant bytes = 3*4 + 2*2 + 1*0 = 16 bytes


## 3. Bounded streams at both levels

The outer stream honors `limit(2)` and each handle's member stream honors `member_limit(2)`. Handles expose complete metrics but do not contain an eager `members` tuple.

In [ ]:
group_count = member_count = 0
with closing(query.stream()) as groups:
    for group in groups:
        group_count += 1
        with closing(group.stream_members()) as members:
            member_count += sum(1 for _ in members)
assert (group_count, member_count) == (2, 4)

## 4. Remove group and member bounds explicitly

Names containing `all` remove only their corresponding bound. SQLite may aggregate and sort before the first yield; the guarantee is incremental Python materialization, not constant database-engine memory.

In [ ]:
group_count = member_count = redundant_bytes = 0
with closing(query.stream_all()) as groups:
    for group in groups:
        group_count += 1
        redundant_bytes += group.potential_redundant_bytes
        with closing(group.stream_all_members()) as members:
            member_count += sum(1 for _ in members)
assert (group_count, member_count, redundant_bytes) == (3, 9, 16)

# here we get only the 1st two groups and fetch all members (3+4=7)
with closing(query.stream()) as groups:
    complete_members_in_selected_groups = sum(
        sum(1 for _ in group.stream_all_members()) for group in groups
    )
assert complete_members_in_selected_groups == 7

## 5. Typed sorting and early closure

Sort by content size, complete file count, or complete redundant bytes. Repeated `sort()` calls replace the earlier ordering. Full content identity breaks ties ascending, and member paths are always ascending.

In [ ]:
by_count = query.sort(duplicate.file_count.desc())
by_redundancy = query.sort(duplicate.redundant_bytes.asc())
smallest_first = by_count.sort(size.asc())
with closing(smallest_first.stream_all()) as groups:
    assert next(groups).size_bytes == 0
assert [group.potential_redundant_bytes for group in by_redundancy.stream_all()] == [0, 4, 12]

with closing(smallest_first.stream_all()) as groups:
    first_group = next(groups)
    with closing(first_group.stream_all_members()) as members:
        first_path = next(members).relative_path
assert first_path.as_posix() == "empty/0.bin"

## 6. Exercise

Predict how many files are visited when every group is streamed but at most one member per group is requested. Does each handle's complete file count change?

In [ ]:
visited = complete_file_count = 0
with closing(query.member_limit(1).stream_all()) as groups:
    for group in groups:
        complete_file_count += group.file_instance_count
        with closing(group.stream_members()) as members:
            visited += sum(1 for _ in members)
assert visited == 3
assert complete_file_count == 9

## 7. Lifetime, pitfalls, and cleanup

A group stream pins one successful scan on first advancement, and its handles retain that scan after refresh or outer-stream closure. Keep the catalog open until all member work finishes. A plain `break` does not guarantee cleanup, so use `contextlib.closing` for partial consumption. These observations do not authorize file mutation; later workflows must validate live bytes and define recovery.

In [ ]:
catalog.close()
workspace.cleanup()